# Baseline Models: SARIMA & Prophet

先用统计模型跑个baseline，后面再试深度学习的。

- Train: 2015-2017
- Test: 2018

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.arima_model import SarimaForecaster, naive_forecast, seasonal_naive
from src.models.prophet_model import ProphetForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, save_figure

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)
print(f'Train: {len(train)} months ({train.index[0].strftime("%Y-%m")} ~ {train.index[-1].strftime("%Y-%m")})')
print(f'Test: {len(test)} months ({test.index[0].strftime("%Y-%m")} ~ {test.index[-1].strftime("%Y-%m")})')

## 1. Naive Baselines

最简单的baseline，总得有个参照物

In [ ]:
naive_pred = naive_forecast(train, test)
snaive_pred = seasonal_naive(train, test)

results = []
results.append(evaluate_forecast(test, naive_pred, 'Naive'))
results.append(evaluate_forecast(test, snaive_pred, 'Seasonal Naive'))

print('Naive:', results[0])
print('Seasonal Naive:', results[1])

In [ ]:
fig = plot_forecast(train, test, snaive_pred, 'Seasonal Naive Forecast')
plt.show()

Seasonal naive的MAPE还行，说明季节性pattern确实很强。

## 2. SARIMA

用auto_arima自动选参数

In [ ]:
sarima = SarimaForecaster()
sarima.auto_fit(train, seasonal=True, m=12)

In [ ]:
sarima_pred = sarima.predict(steps=len(test))
sarima_pred.index = test.index

results.append(evaluate_forecast(test, sarima_pred, 'SARIMA'))
print('SARIMA:', results[-1])

In [ ]:
fig = plot_forecast(train, test, sarima_pred, 'SARIMA Forecast')
save_figure(fig, 'sarima_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
# 看看残差
print(sarima.get_summary())

## 3. Prophet

Facebook Prophet，自带holiday效应处理

In [ ]:
prophet = ProphetForecaster(
    yearly_seasonality=True,
    changepoint_prior_scale=0.05
)
prophet.fit(train)

In [ ]:
prophet_pred = prophet.predict(periods=len(test))
prophet_pred.index = test.index

results.append(evaluate_forecast(test, prophet_pred.values, 'Prophet'))
print('Prophet:', results[-1])

In [ ]:
fig = plot_forecast(train, test, prophet_pred.values, 'Prophet Forecast')
save_figure(fig, 'prophet_forecast.png', output_dir='../results/figures')
plt.show()

In [ ]:
# prophet的components
forecast_df = prophet.get_components(periods=len(test))
fig = prophet.model.plot_components(forecast_df)
plt.tight_layout()
plt.show()

## 4. 对比

In [ ]:
comparison = compare_models(results)
print(comparison.to_string())

In [ ]:
# 画一下所有预测的对比
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train.values, label='Train', color='gray', alpha=0.5)
ax.plot(test.index, test.values, label='Actual', color='black', linewidth=2)
ax.plot(test.index, sarima_pred.values, label='SARIMA', linestyle='--')
ax.plot(test.index, prophet_pred.values, label='Prophet', linestyle='--')
ax.plot(test.index, snaive_pred.values, label='Seasonal Naive', linestyle=':')
ax.legend()
ax.set_title('All Models Comparison')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/figures/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

SARIMA和Prophet都比naive好不少。下一步试试LSTM能不能再提升。